<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/alex/RA1_TP4/RA1_Trabajo_pr%C3%A1ctico_N%C2%B0_4_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#TP4 - Aprendizaje no supervisado

In [1]:
# @title *Esta celda importa utilidades comunes al colab*
import sys
import os

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import warnings
from scipy.stats import norm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error, r2_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier


#Ejercicio 1 - Triaje oncológico asistido por algoritmos


---
**Datasets:** `datos_pacientes.csv` y `datos_pacientes_diagnosticados.csv`

**Grupo:** Numero 8  
**Integrantes:** Acosta Alex, Bareiro Santiago, Borges Agustin  
**Fecha:** x/04/2026

>Un hospital regional de mediana complejidad atiende una cantidad creciente de pacientes derivados con
sospecha de tumor mamario. El cuello de botella no es la tecnología de diagnóstico (el hospital dispone
de equipamiento para biopsia digital) sino la disponibilidad del personal médico especializado: por
restricciones de recursos humanos, el equipo de oncología puede analizar y emitir diagnóstico definitivo
(benigno o maligno) para no más de 2 pacientes por día.
Actualmente hay 560 pacientes con estudios preliminares ya realizados en el sistema, esperando
diagnóstico. A ese ritmo, el último paciente de la lista esperaría 280 días. Para tumores malignos, esa
demora puede ser la diferencia entre un tratamiento efectivo y uno tardío.
El jefe del servicio de oncología propone implementar un sistema de triaje asistido por algoritmos: usar los
valores de los estudios preliminares disponibles para todos los pacientes (datos_pacientes.csv) para
identificar automáticamente cuáles tienen mayor probabilidad de tumor maligno y priorizarlos en la
agenda. Para calibrar y validar el sistema, el servicio dispone de un conjunto reducido de pacientes ya
diagnosticados por el equipo médico (datos_pacientes_diagnosticados.csv), con su resultado confirmado.
El dataset de estudios contiene 30 características computadas a partir de imágenes digitalizadas de
biopsias. El conjunto diagnosticado incluye, además, la columna diagnosis (M = maligno, B = benigno).
El jefe del servicio no rechazará el sistema si tiene un error ocasional. Lo rechazará si no puede entender
qué hace, si produce resultados inconsistentes o si no puede cuantificar el riesgo de que un paciente
maligno quede al final de la lista.





---


## 0. Configuración del entorno

In [2]:
# Dataset desde el repositorio en Drive

!wget -c --no-check-certificate "https://drive.google.com/uc?export=download&id=1CN6_hIjs4JFsJxnyijZmvAE0xcoyMONq&confirm=t" -O datos_pacientes.zip

--2026-04-22 17:20:33--  https://drive.google.com/uc?export=download&id=1CN6_hIjs4JFsJxnyijZmvAE0xcoyMONq&confirm=t
Resolving drive.google.com (drive.google.com)... 173.194.203.138, 173.194.203.102, 173.194.203.100, ...
Connecting to drive.google.com (drive.google.com)|173.194.203.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1CN6_hIjs4JFsJxnyijZmvAE0xcoyMONq&export=download [following]
--2026-04-22 17:20:33--  https://drive.usercontent.google.com/download?id=1CN6_hIjs4JFsJxnyijZmvAE0xcoyMONq&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.195.132, 2607:f8b0:400e:c17::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.195.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 50200 (49K) [application/octet-stream]
Saving to: ‘datos_pacientes.zip’

datos_pacientes.zip 100%[===================>]  

In [3]:
!unzip datos_pacientes.zip

Archive:  datos_pacientes.zip
  inflating: datos_pacientes_diagnosticados.csv  
  inflating: datos_pacientes.csv     


### 0.1 EDA

In [6]:
dataset_pacientes = pd.read_csv('datos_pacientes.csv')
dataset_pacientes_diagnosticados = pd.read_csv('datos_pacientes_diagnosticados.csv')

Se muestran los primeros 5 pacientes del dataset

In [7]:
dataset_pacientes.head()

,id,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,915940,14.580,13.66,94.29,658.8,0.09832,0.08918,0.08222,0.043490,0.1739,...,16.76,17.24,108.50,862.0,0.1223,0.1928,0.24920,0.09186,0.2626,0.07048
1,893988,11.540,10.72,73.73,409.1,0.08597,0.05969,0.01367,0.008907,0.1833,...,12.34,12.87,81.23,467.8,0.1092,0.1626,0.08324,0.04715,0.3390,0.07434
2,8910251,10.600,18.95,69.28,346.4,0.09688,0.11470,0.06387,0.026420,0.1922,...,11.88,22.94,78.28,424.8,0.1213,0.2515,0.19160,0.07926,0.2940,0.07587
3,8510824,9.504,12.44,60.34,273.9,0.10240,0.06492,0.02956,0.020760,0.1815,...,10.23,15.66,65.13,314.9,0.1324,0.1148,0.08867,0.06227,0.2450,0.07773
4,90944601,13.780,15.79,88.37,585.9,0.08817,0.06718,0.01055,0.009937,0.1405,...,15.27,17.50,97.90,706.6,0.1072,0.1071,0.03517,0.03312,0.1859,0.06810


In [8]:
print(f"El dataset_pacientes posee {len(dataset_pacientes)} índices (filas).")

El dataset_pacientes posee 560 índices (filas).


In [9]:
print('Null values in dataset_pacientes:')
print(dataset_pacientes.isnull().sum())

Null values in dataset_pacientes:
id                         0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64


In [10]:
dataset_pacientes_diagnosticados

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,898677,B,10.260,14.71,66.20,321.6,0.09882,0.09159,0.035810,0.020370,...,10.880,19.48,70.89,357.1,0.1360,0.1636,0.07162,0.04074,0.2434,0.08488
1,913505,M,19.440,18.82,128.10,1167.0,0.10890,0.14480,0.225600,0.119400,...,23.960,30.39,153.90,1740.0,0.1514,0.3725,0.59360,0.20600,0.3266,0.09009
2,912519,B,13.470,14.06,87.32,546.3,0.10710,0.11550,0.057860,0.052660,...,14.830,18.32,94.94,660.2,0.1393,0.2499,0.18480,0.13350,0.3227,0.09326
3,84667401,M,13.730,22.61,93.60,578.3,0.11310,0.22930,0.212800,0.080250,...,15.030,32.01,108.80,697.7,0.1651,0.7725,0.69430,0.22080,0.3596,0.14310
4,874158,B,10.080,15.11,63.76,317.5,0.09267,0.04695,0.001597,0.002404,...,11.870,21.18,75.39,437.0,0.1521,0.1019,0.00692,0.01042,0.2933,0.07697
5,90524101,M,17.990,20.66,117.80,991.7,0.10360,0.13040,0.120100,0.088240,...,21.080,25.41,138.10,1349.0,0.1482,0.3735,0.33010,0.19740,0.3060,0.08503
6,858477,B,8.618,11.79,54.34,224.5,0.09752,0.05272,0.020610,0.007799,...,9.507,15.40,59.90,274.9,0.1733,0.1239,0.11680,0.04419,0.3220,0.09026
7,865432,B,14.500,10.89,94.28,640.7,0.11010,0.10990,0.088420,0.057780,...,15.700,15.98,102.80,745.5,0.1313,0.1788,0.25600,0.12210,0.2889,0.08006
8,857155,B,12.050,14.63,78.04,449.3,0.10310,0.09092,0.065920,0.027490,...,13.760,20.70,89.88,582.6,0.1494,0.2156,0.30500,0.06548,0.2747,0.08301


In [ ]:
print(f"El dataset_pacientes posee {len(dataset_pacientes)} índices (filas).")

El dataset_pacientes posee 560 índices (filas).


In [ ]:
print('Null values in dataset_pacientes:')
print(dataset_pacientes.isnull().sum())

Null values in dataset_pacientes:
id                         0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64




---


##1. Construcción y ensamblado del pipeline



---


##2. Entrenamiento y comparación de modelos



---


## 3. Optimización

### 3.1 Selección del Modelo y Búsqueda de Hiperparámetros



### 3.2 Análisis de Métricas y Comparación de Modelos

### 3.3 Identificación del modelo con la mejor métrica:

### 3.5 Entrenamiento del mejor modelo con mas y menos features



### 3.6 Grafica de los residuos del mejor modelo



---


## 4. Conclusiones

